In [ ]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数 — SAE 稀疏自编码器因子（已训练模型，仅推理 + 正交化）

    返回:
        pd.DataFrame: 包含 date, instrument, factor 三列，无 inf
    """
    import io, base64, time
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import dai
    import structlog

    logger = structlog.get_logger()

    # ============================================================
    # 0. 嵌入模型权重（base64 字符串 —— 请替换为你的实际文件内容）
    # ============================================================
    WEIGHTS_B64 = (
        # 用你的 B_g013_b64.txt 内容替换这里的占位符
        "..."
    )

    # ============================================================
    # 1. 常量 & 模型架构（从 sae.py / config.py 提取）
    # ============================================================
    BAR_FEATURES = [
        "ret", "hl_range", "close_vwap_gap", "rel_vol", "spread",
        "depth_imb1", "depth_imb5", "ord_imb", "microprice_tilt",
        "signed_vol", "avg_trade_size", "rv_1min"
    ]
    BARS_PER_DAY = 48
    BAR_DOWNSAMPLE = 5
    TIME_BLOCKS = 6
    DAILY_FEATURES = [
        "netflow_amount_main", "netflow_amount_rate_main",
        "net_active_buy_amount_main", "reversal_5", "momentum_5", "turn"
    ]

    # 正交化使用的 36 个因子库列名（平台 bigalpha_2026_factorlib 实际字段）
    LIB_COLUMNS = [
        "close", "volume", "amount", "turn", "change_ratio", "daily_return",
        "momentum_5", "reversal_5", "volatility_5",
        "total_market_cap", "float_market_cap", "pe_ttm", "pb", "ps_ttm",
        "sma_20", "ema_20",
        "macd_diff_12_26_9", "macd_dea_12_26_9", "macd_hist_12_26_9",
        "rsi_12", "kdj_k_9_3_3", "kdj_d_9_3_3", "bias_20", "cci_14", "atr_14",
        "roe_avg_ttm", "roa_avg_ttm", "gross_profit_rate_ttm", "net_profit_rate_ttm",
        "debt_to_asset_lf", "current_ratio_lf",
        "netflow_amount_main", "netflow_amount_rate_main", "net_active_buy_amount_main",
        "beta_000300SH_22", "list_days"
    ]

    # ---------------------- 模型定义 ----------------------
    class SparseLinear(nn.Module):
        def __init__(self, in_f, out_f, density):
            super().__init__()
            self.in_f, self.out_f = in_f, out_f
            self.weight = nn.Parameter(torch.empty(out_f, in_f))
            self.bias = nn.Parameter(torch.zeros(out_f))
            nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
            self.register_buffer("allowed", torch.ones(out_f, in_f))
            self.register_buffer("mask", self._random_mask(density))

        def _random_mask(self, density):
            allowed_flat = self.allowed.view(-1)
            cand = allowed_flat.nonzero(as_tuple=True)[0]
            k = min(int(round(density * len(cand))), len(cand))
            m = torch.zeros(self.out_f * self.in_f)
            if k > 0:
                m[cand[torch.randperm(len(cand))[:k]]] = 1.0
            return m.view(self.out_f, self.in_f)

        @torch.no_grad()
        def set_allowed(self, allowed):
            assert allowed.shape == (self.out_f, self.in_f)
            self.allowed.copy_(allowed.float())
            self.mask.mul_(self.allowed)
            self.weight.mul_(self.allowed)

        def forward(self, x):
            return F.linear(x, self.weight * self.mask, self.bias)

    class Genome:
        def __init__(self, in_dim, hidden, init_density, alpha, beta, gamma,
                     lr, weight_decay, total_epochs, prune_epochs, rigl_interval,
                     rigl_drop_frac, adapt_epochs, adapt_lr, adapt_tail_days,
                     dropout, rank_target, train_years, seed=0,
                     feat_mask=None, time_mask=None, daily_mask=None):
            self.in_dim = in_dim
            self.hidden = hidden
            self.init_density = init_density
            self.alpha = alpha
            self.beta = beta
            self.gamma = gamma
            self.lr = lr
            self.weight_decay = weight_decay
            self.total_epochs = total_epochs
            self.prune_epochs = prune_epochs
            self.rigl_interval = rigl_interval
            self.rigl_drop_frac = rigl_drop_frac
            self.adapt_epochs = adapt_epochs
            self.adapt_lr = adapt_lr
            self.adapt_tail_days = adapt_tail_days
            self.dropout = dropout
            self.rank_target = rank_target
            self.train_years = train_years
            self.seed = seed
            self.feat_mask = feat_mask
            self.time_mask = time_mask
            self.daily_mask = daily_mask
            self._visible = None

        def visible(self):
            if self._visible is not None:
                return self._visible
            F_ = len(BAR_FEATURES)
            n_bars = BARS_PER_DAY
            intraday = n_bars * F_
            vis = np.zeros(self.in_dim, dtype=bool)
            fm = np.ones(F_, bool) if self.feat_mask is None else np.asarray(self.feat_mask, bool)
            tm = np.ones(TIME_BLOCKS, bool) if self.time_mask is None else np.asarray(self.time_mask, bool)
            per_block = max(1, n_bars // TIME_BLOCKS)
            for b in range(min(n_bars, self.in_dim // max(F_, 1))):
                if not tm[min(b // per_block, TIME_BLOCKS - 1)]:
                    continue
                for f in range(F_):
                    p = b * F_ + f
                    if p < intraday and p < self.in_dim and fm[f]:
                        vis[p] = True
            n_daily = max(0, self.in_dim - intraday)
            if n_daily:
                dm = np.ones(n_daily, bool) if self.daily_mask is None else np.asarray(self.daily_mask, bool)[:n_daily]
                vis[intraday:intraday + n_daily] = dm
            if not vis.any():
                vis[:] = True
            self._visible = vis
            return vis

    class SAE(nn.Module):
        def __init__(self, g):
            super().__init__()
            self.g = g
            torch.manual_seed(g.seed)
            vis = torch.tensor(g.visible(), dtype=torch.bool)
            self.register_buffer("vis", vis)
            self.register_buffer("vis_f", vis.float())
            self.enc = SparseLinear(g.in_dim, g.hidden, g.init_density)
            self.dec = SparseLinear(g.hidden, g.in_dim, g.init_density)
            self.enc.set_allowed(vis.unsqueeze(0).expand(g.hidden, g.in_dim).float())
            self.dec.set_allowed(vis.unsqueeze(1).expand(g.in_dim, g.hidden).float())
            self.head = nn.Linear(g.hidden, 1)
            self.drop = nn.Dropout(g.dropout)

        def forward(self, x):
            x = x * self.vis_f
            z = F.relu(self.enc(x))
            z = self.drop(z)
            return self.head(z).squeeze(-1)

    # ---------------------- 特征工程函数 ----------------------
    def _per_bar_features(g, cumulative=True):
        g = g.sort_values("date").reset_index(drop=True)
        n = len(g)
        if n == 0:
            return pd.DataFrame({"instrument": "", "day": pd.NaT, "bar_idx": range(BARS_PER_DAY),
                                 **{feat: 0.0 for feat in BAR_FEATURES}})
        if cumulative:
            vol = np.diff(g["volume"].to_numpy(float), prepend=0.0)
            amt = np.diff(g["amount"].to_numpy(float), prepend=0.0)
            deals = np.diff(g["deal_number"].to_numpy(float), prepend=0.0)
        else:
            vol = g["volume"].to_numpy(float)
            amt = g["amount"].to_numpy(float)
            deals = g["deal_number"].to_numpy(float)
        vol = np.clip(vol, 0, None)
        amt = np.clip(amt, 0, None)
        deals = np.clip(deals, 0, None)

        close = g["close"].to_numpy(float)
        open_ = g["open"].to_numpy(float)
        high = g["high"].to_numpy(float)
        low = g["low"].to_numpy(float)

        ret = np.empty(n)
        ret[0] = (close[0] / open_[0] - 1.0) if open_[0] != 0 else 0.0
        ret[1:] = close[1:] / np.where(close[:-1] == 0, np.nan, close[:-1]) - 1.0
        ret = np.nan_to_num(ret)

        vwap = np.where(vol > 0, amt / np.where(vol == 0, np.nan, vol), close)
        hl_range = np.where(close != 0, (high - low) / close, 0.0)
        close_vwap_gap = np.where(close != 0, (close - vwap) / close, 0.0)
        mean_vol = vol.mean() if vol.mean() > 0 else 1.0
        rel_vol = vol / mean_vol

        a1 = g["ask_price1"].to_numpy(float)
        b1 = g["bid_price1"].to_numpy(float)
        mid = (a1 + b1) / 2.0
        spread = np.where(mid > 0, (a1 - b1) / mid, 0.0)

        bv1 = g["bid_volume1"].to_numpy(float)
        av1 = g["ask_volume1"].to_numpy(float)
        denom1 = bv1 + av1
        depth_imb1 = np.where(denom1 > 0, (bv1 - av1) / denom1, 0.0)

        bv5 = g[[f"bid_volume{i}" for i in range(1, 6)]].to_numpy(float).sum(1)
        av5 = g[[f"ask_volume{i}" for i in range(1, 6)]].to_numpy(float).sum(1)
        denom5 = bv5 + av5
        depth_imb5 = np.where(denom5 > 0, (bv5 - av5) / denom5, 0.0)

        bn = g[[f"bid_num_orders{i}" for i in range(1, 6)]].to_numpy(float).sum(1)
        an = g[[f"ask_num_orders{i}" for i in range(1, 6)]].to_numpy(float).sum(1)
        denom_n = bn + an
        ord_imb = np.where(denom_n > 0, (bn - an) / denom_n, 0.0)

        micro = np.where(denom1 > 0, (a1 * bv1 + b1 * av1) / denom1, mid)
        microprice_tilt = np.where(mid > 0, (micro - mid) / mid, 0.0)
        signed_vol = rel_vol * np.sign(ret)
        avg_trade_size = np.where(deals > 0, amt / deals, np.nan)

        B = BAR_DOWNSAMPLE
        K = BARS_PER_DAY
        n_full_buckets = n // B
        if n_full_buckets == 0:
            return pd.DataFrame({"instrument": g["instrument"].iloc[0],
                                 "day": g["date"].iloc[0].normalize(),
                                 "bar_idx": range(K),
                                 **{feat: 0.0 for feat in BAR_FEATURES}})

        total_day_vol = vol.sum()
        mean_bucket_vol = total_day_vol / n_full_buckets if n_full_buckets > 0 else 1.0
        bucket_data = []
        for j in range(n_full_buckets):
            s = j * B; e = s + B
            bucket_close = close[e-1]
            prev_close = open_[0] if j == 0 else close[s-1]
            bucket_ret = (bucket_close / prev_close - 1.0) if prev_close != 0 else 0.0
            bucket_high = high[s:e].max(); bucket_low = low[s:e].min()
            bucket_hl_range = (bucket_high - bucket_low) / bucket_close if bucket_close != 0 else 0.0
            bucket_vol = vol[s:e].sum(); bucket_amt = amt[s:e].sum()
            bucket_vwap = bucket_amt / bucket_vol if bucket_vol > 0 else bucket_close
            bucket_close_vwap_gap = (bucket_close - bucket_vwap) / bucket_close if bucket_close != 0 else 0.0
            bucket_rel_vol = bucket_vol / mean_bucket_vol
            bucket_spread = np.mean(spread[s:e])
            bucket_depth_imb1 = np.mean(depth_imb1[s:e])
            bucket_depth_imb5 = np.mean(depth_imb5[s:e])
            bucket_ord_imb = np.mean(ord_imb[s:e])
            bucket_microprice_tilt = np.mean(microprice_tilt[s:e])
            bucket_signed_vol = np.mean(signed_vol[s:e])
            bucket_avg_trade_size = np.nanmean(avg_trade_size[s:e])
            bucket_rv_1min = np.std(ret[s:e], ddof=1) if (e - s) > 1 else 0.0
            bucket_data.append([bucket_ret, bucket_hl_range, bucket_close_vwap_gap, bucket_rel_vol,
                                bucket_spread, bucket_depth_imb1, bucket_depth_imb5, bucket_ord_imb,
                                bucket_microprice_tilt, bucket_signed_vol,
                                bucket_avg_trade_size, bucket_rv_1min])
        if len(bucket_data) < K:
            pad = bucket_data[0] if bucket_data else [0.0]*len(BAR_FEATURES)
            bucket_data += [list(pad)] * (K - len(bucket_data))
        bucket_data = bucket_data[:K]
        arr = np.array(bucket_data, dtype=float)
        out = pd.DataFrame({"instrument": g["instrument"].iloc[0],
                            "day": g["date"].iloc[0].normalize(),
                            "bar_idx": np.arange(K)})
        for i, feat in enumerate(BAR_FEATURES):
            out[feat] = arr[:, i]
        return out

    def assemble_matrix(panel, features=None):
        features = features or BAR_FEATURES
        panel = panel.sort_values(["day", "instrument", "bar_idx"])
        keys = panel[["day", "instrument"]].drop_duplicates().reset_index(drop=True)
        F = len(features)
        expected = len(keys) * BARS_PER_DAY
        assert len(panel) == expected, f"Panel shape mismatch: {len(panel)} vs {expected}"
        arr = panel[features].to_numpy(float).reshape(len(keys), BARS_PER_DAY * F)
        idx = keys.rename(columns={"day": "date"})
        return arr, idx

    def _standardize_xs(X, day_ids):
        Xs = X.astype(np.float64).copy()
        for d in np.unique(day_ids):
            m = day_ids == d
            blk = Xs[m]
            mu = blk.mean(0); sd = blk.std(0); sd[sd == 0] = 1.0
            Xs[m] = (blk - mu) / sd
        return np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)

    def predict(model, X, groups):
        model.eval()
        out = np.full(X.shape[0], np.nan, dtype=np.float64)
        with torch.no_grad():
            for idx in groups:
                y_hat = model(torch.as_tensor(X[idx], dtype=torch.float32))
                out[idx] = y_hat.cpu().numpy()
        return out

    def orthogonalize(scores_df, lib_df, lib_cols):
        m = scores_df.merge(lib_df, on=["date", "instrument"], how="left")
        out_parts = []
        for _, g in m.groupby("date", sort=True):
            s = g["score"].to_numpy(float)
            L = g[lib_cols].to_numpy(float)
            ok = np.isfinite(s) & np.isfinite(L).all(axis=1)
            res = np.full(len(g), np.nan)
            if ok.sum() > len(lib_cols) + 5:
                Lo = np.column_stack([np.ones(ok.sum()), L[ok]])
                beta, *_ = np.linalg.lstsq(Lo, s[ok], rcond=1e-10)
                res[ok] = s[ok] - Lo @ beta
            gg = g[["date", "instrument"]].copy()
            gg["score"] = res
            out_parts.append(gg)
        return pd.concat(out_parts, ignore_index=True).dropna(subset=["score"]).reset_index(drop=True)

    # ============================================================
    # 2. 实际推理流程
    # ============================================================
    logger.info("开始SAE因子推理", test_start=str(start_date), test_end=str(end_date))

    # 2.1 加载模型
    t0 = time.time()
    ck = torch.load(io.BytesIO(base64.b64decode(WEIGHTS_B64)), map_location="cpu")
    gd = dict(ck["genome"])
    for k in ("feat_mask", "time_mask", "daily_mask"):
        if isinstance(gd.get(k), list):
            gd[k] = tuple(gd[k])
    genome_args = {k: v for k, v in gd.items() if k in Genome.__init__.__code__.co_varnames}
    g = Genome(**genome_args)
    model = SAE(g)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    logger.info(f"模型加载完成, hidden={g.hidden}, in_dim={g.in_dim}, {time.time()-t0:.1f}s")

    # 2.2 从 bar1m 构建特征矩阵
    bar_table = datasources["bar1m"]
    s = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    e = pd.Timestamp(end_date).strftime("%Y-%m-%d")

    raw = dai.query(
        f"""SELECT date, instrument,
                   open, high, low, close, volume, amount, deal_number,
                   ask_price1, bid_price1,
                   ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5,
                   bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5,
                   ask_num_orders1, ask_num_orders2, ask_num_orders3, ask_num_orders4, ask_num_orders5,
                   bid_num_orders1, bid_num_orders2, bid_num_orders3, bid_num_orders4, bid_num_orders5
            FROM {bar_table}
            WHERE date BETWEEN '{s}' AND '{e}'
        """,
        filters={"date": [s, e]}
    ).df()
    raw["date"] = pd.to_datetime(raw["date"])
    raw["day"] = raw["date"].dt.normalize()
    logger.info(f"原始分钟数据: {len(raw)} rows")

    t1 = time.time()
    parts = []
    for (inst, day), grp in raw.groupby(["instrument", "day"], sort=False):
        parts.append(_per_bar_features(grp, cumulative=True))
    panel = pd.concat(parts, ignore_index=True)
    logger.info(f"特征面板构建: {len(panel)} rows, {time.time()-t1:.1f}s")

    X, idx = assemble_matrix(panel)
    idx = idx.reset_index(drop=True)

    # 日度因子
    daily_df = dai.query(
        f"SELECT date, instrument, {', '.join(DAILY_FEATURES)} FROM bigalpha_2026_factorlib",
        filters={"date": [s, e]}
    ).df()
    daily_df["date"] = pd.to_datetime(daily_df["date"]).dt.normalize()
    merged = idx.merge(daily_df, on=["date", "instrument"], how="left")
    for c in DAILY_FEATURES:
        if c not in merged.columns:
            merged[c] = 0.0
    Xd = np.nan_to_num(merged[DAILY_FEATURES].to_numpy(float), nan=0.0, posinf=0.0, neginf=0.0)
    Xall = np.hstack([X, Xd])

    assert Xall.shape[1] == g.in_dim, f"特征维度不匹配: {Xall.shape[1]} vs 模型要求 {g.in_dim}"

    day_ids = merged["date"].astype("category").cat.codes.to_numpy()
    Xs = _standardize_xs(Xall, day_ids)
    groups = [np.where(day_ids == d)[0] for d in np.unique(day_ids)]

    # 推理
    raw_scores = predict(model, Xs.astype(np.float32), groups)
    out = merged[["date", "instrument"]].copy()
    out["score"] = raw_scores
    out = out.dropna(subset=["score"]).reset_index(drop=True)
    logger.info(f"原始因子: {len(out)} 行")

    # 正交化
    lib = dai.query("SELECT * FROM bigalpha_2026_factorlib", filters={"date": [s, e]}).df()
    lib["date"] = pd.to_datetime(lib["date"]).dt.normalize()
    # 使用平台实际的 36 列
    ortho = orthogonalize(out, lib, LIB_COLUMNS)
    result = ortho.rename(columns={"score": "factor"})[["date", "instrument", "factor"]]
    logger.info(f"正交化因子: {len(result)} 行")

    # 2.3 对齐 CSI1000 成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]}
    ).df()
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    result = pd.merge(result, stk_pool, how="inner", on=["date", "instrument"])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["factor"]).reset_index(drop=True)[["date", "instrument", "factor"]]

    logger.info(f"最终因子: {len(result)} 行, 耗时 {time.time()-t0:.1f}s")
    return result